# Rizzvision v5 — Multi-Label Clothing Classifier

**Model:** EfficientNetB3 + sigmoid head (multi-label)  
**Classes:** tops · bottoms · footwear · outerwear · dress  
**Input:** full scene images — NO cropping  
**Loss:** Class-weighted binary cross-entropy (outerwear 2.5×, footwear 1.8×)  
**Target:** TPR ≥ 92% per class at FPR ≤ 2%  
**Output:** `clothing_classifier_v5.keras` + `thresholds_v5.json`

### Required input datasets
Add ALL of the following before running (CSVs reference their original paths):
- `nitin2807/rv-dataset-v5` ← label CSVs only, no images
- `nitin2807/deepfashion2-full`
- `nitin2807/fashionpedia-full`
- `nitin2807/imaterialist-fashion-2020`
- `aryashah2k/large-shoe-dataset-ut-zappos50k`
- `noobyogi0100/shoe-dataset`
- `hasibalmuzdadid/shoe-vs-sandal-vs-boot-dataset-15k-images`
- `iaasileperuma/footwear-images-dataset`
- `kritanjalijain/outfititems`
- `paramaggarwal/fashion-product-images-dataset`
- `agrigorev/clothing-dataset-full`
- `silverstone1903/deep-fashion-multimodal`

In [ ]:
# ── Cell 0: Config ────────────────────────────────────────────────────────────
DATASET_DIR   = '/kaggle/input/datasets/nitin2807/rv-dataset-v5'
OUT_DIR       = '/kaggle/working'

LABELS        = ['tops', 'bottoms', 'footwear', 'outerwear', 'dress']
NUM_CLASSES   = len(LABELS)
IMG_SIZE      = 300          # EfficientNetB3 native
BATCH_SIZE    = 32           # P100 safe; raise to 48 on A100

# Phase 1 — train head only
PHASE1_EPOCHS = 5
PHASE1_LR     = 1e-3

# Phase 2 — fine-tune top N layers of backbone
PHASE2_EPOCHS    = 60
PHASE2_LR_MAX    = 1e-4
PHASE2_LR_MIN    = 1e-6
UNFREEZE_LAYERS  = 150
LABEL_SMOOTHING  = 0.05
PATIENCE         = 8

# Class weights — boost recall for data-starved / harder classes
CLASS_WEIGHTS = {
    'tops':      1.0,
    'bottoms':   1.0,
    'footwear':  1.8,
    'outerwear': 2.5,
    'dress':     1.2,
}

# Resume support
RESUME_FROM_CHECKPOINT = False
CHECKPOINT_PATH        = f'{OUT_DIR}/best_v5.keras'
PHASE2_DONE            = 0

# Threshold calibration — target TPR ≥ 92% with FPR ≤ 2%
MIN_THRESHOLD  = 0.25
TARGET_TPR     = 0.92
MAX_FPR        = 0.020

import os, math, json, random
import numpy as np
import pandas as pd
import tensorflow as tf
print('TF:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Class weights:', CLASS_WEIGHTS)

In [ ]:
# ── Cell 1: Runtime check ─────────────────────────────────────────────────────
# (Keras 3 / TF 2.16+ no longer needs the deserialisation patch)
import keras
print('Keras:', keras.__version__)

In [ ]:
# ── Cell 2: Dataset pipeline ──────────────────────────────────────────────────
from tensorflow.keras.applications.efficientnet import preprocess_input

def load_csv(split):
    path = os.path.join(DATASET_DIR, f'labels_{split}.csv')
    df = pd.read_csv(path)
    # Paths are absolute /kaggle/input/... source paths — use them directly.
    # Verify a sample exists to catch mount issues early.
    sample = df['filepath'].iloc[0]
    if not os.path.exists(sample):
        raise FileNotFoundError(
            f'Sample image not found: {sample}\n'
            'Ensure all source datasets are added as inputs to this notebook.'
        )
    return df

df_train = load_csv('train')
df_val   = load_csv('val')
df_test  = load_csv('test')
print(f'train={len(df_train):,}  val={len(df_val):,}  test={len(df_test):,}')
print('Label distribution (train):')
print(df_train[LABELS].sum())

# ─── Augmentation ─────────────────────────────────────────────────────────────
def augment(image):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.15)
    image = tf.image.random_contrast(image, 0.85, 1.15)
    image = tf.image.random_saturation(image, 0.85, 1.15)
    angle = tf.random.uniform([], -0.26, 0.26)
    image = _rotate(image, angle)
    return image

def _rotate(image, angle):
    try:
        import tensorflow_addons as tfa
        return tfa.image.rotate(image, angle, interpolation='BILINEAR')
    except ImportError:
        return image

def read_image(filepath, label, training=False):
    raw = tf.io.read_file(filepath)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)  # handles JPEG/PNG/GIF/BMP
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    if training:
        img = augment(img)
    img = preprocess_input(img)
    return img, label

def make_dataset(df, training=False):
    filepaths = df['filepath'].values
    labels    = df[LABELS].values.astype('float32')
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if training:
        ds = ds.shuffle(min(10000, len(df)), reshuffle_each_iteration=True)
    ds = ds.map(
        lambda fp, lbl: read_image(fp, lbl, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, training=True)
val_ds   = make_dataset(df_val,   training=False)
test_ds  = make_dataset(df_test,  training=False)
print('Datasets ready')

In [ ]:
# ── Cell 3: Build or load model ───────────────────────────────────────────────
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3

def build_model():
    base = EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False  # Phase 1: frozen backbone
    inp  = base.input
    x    = base.output
    x    = layers.GlobalAveragePooling2D()(x)
    x    = layers.BatchNormalization()(x)
    x    = layers.Dense(512, activation='relu',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x    = layers.Dropout(0.4)(x)
    x    = layers.Dense(256, activation='relu',
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x    = layers.Dropout(0.3)(x)
    # Sigmoid output — one probability per label, independent
    out  = layers.Dense(NUM_CLASSES, activation='sigmoid', name='predictions')(x)
    return Model(inp, out)

if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    print(f'Loading checkpoint: {CHECKPOINT_PATH}')
    model = tf.keras.models.load_model(CHECKPOINT_PATH)
else:
    model = build_model()
    print('Built fresh model')

model.summary(line_length=100)

In [ ]:
# ── Cell 4: Phase 1 — train head only (with class weights) ───────────────────
if not RESUME_FROM_CHECKPOINT or PHASE2_DONE == 0:
    # Build per-sample weight vector for the loss
    # We pass pos_weight per class to BinaryCrossentropy via sample_weight workaround:
    # use a custom weighted loss instead
    pos_weights = tf.constant([CLASS_WEIGHTS[l] for l in LABELS], dtype=tf.float32)

    def weighted_bce(y_true, y_pred):
        bce = tf.keras.backend.binary_crossentropy(y_true, y_pred)
        # Weight positive examples more heavily per class
        weights = y_true * (pos_weights - 1.0) + 1.0
        return tf.reduce_mean(weights * bce)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(PHASE1_LR),
        loss=weighted_bce,
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='acc', threshold=0.5),
            tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=NUM_CLASSES),
        ]
    )
    print(f'Phase 1: {PHASE1_EPOCHS} epochs, head only')
    print(f'Class weights: {CLASS_WEIGHTS}')
    history1 = model.fit(
        train_ds, epochs=PHASE1_EPOCHS, validation_data=val_ds,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                CHECKPOINT_PATH, save_best_only=True,
                monitor='val_auc', mode='max', verbose=1
            ),
        ]
    )
else:
    print(f'Skipping Phase 1 (resuming at Phase 2 epoch {PHASE2_DONE})')

In [ ]:
# ── Cell 5: Phase 2 — fine-tune top N backbone layers ─────────────────────────
if os.path.exists(CHECKPOINT_PATH):
    model = tf.keras.models.load_model(CHECKPOINT_PATH, custom_objects={'weighted_bce': weighted_bce})
    print('Loaded best checkpoint for Phase 2')

for layer in model.layers:
    layer.trainable = False

pred_idx     = next(i for i, l in enumerate(model.layers) if l.name == 'predictions')
HEAD_BUFFER  = 6
backbone_end = pred_idx - HEAD_BUFFER
print(f'Total layers: {len(model.layers)}  |  backbone end: {backbone_end}')

for layer in model.layers[:backbone_end][-UNFREEZE_LAYERS:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers: {trainable}')

total_steps = PHASE2_EPOCHS * len(train_ds)
steps_done  = PHASE2_DONE  * len(train_ds)
initial_lr  = PHASE2_LR_MIN + 0.5 * (1 + math.cos(math.pi * steps_done / total_steps)) * (PHASE2_LR_MAX - PHASE2_LR_MIN)
print(f'Phase 2 initial LR: {initial_lr:.2e}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(initial_lr),
    loss=weighted_bce,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='acc', threshold=0.5),
        tf.keras.metrics.AUC(name='auc', multi_label=True, num_labels=NUM_CLASSES),
    ]
)

def cosine_lr(epoch, lr):
    step = (PHASE2_DONE + epoch) * len(train_ds)
    return PHASE2_LR_MIN + 0.5 * (1 + math.cos(math.pi * step / total_steps)) * (PHASE2_LR_MAX - PHASE2_LR_MIN)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, save_best_only=True,
        monitor='val_auc', mode='max', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', mode='max',
        patience=PATIENCE, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.LearningRateScheduler(cosine_lr, verbose=0),
]

remaining_epochs = PHASE2_EPOCHS - PHASE2_DONE
print(f'Phase 2: {remaining_epochs} epochs, top {UNFREEZE_LAYERS} backbone layers unfrozen')
history2 = model.fit(train_ds, epochs=remaining_epochs, validation_data=val_ds, callbacks=callbacks)

In [ ]:
# ── Cell 6: Threshold calibration (val set) — F1-optimal per class ────────────
if os.path.exists(CHECKPOINT_PATH):
    model = tf.keras.models.load_model(CHECKPOINT_PATH, custom_objects={'weighted_bce': weighted_bce})
    print('Loaded best checkpoint for calibration')

print('Running inference on val set …')
y_pred_val = model.predict(val_ds, verbose=1)
y_true_val = df_val[LABELS].values.astype('float32')

thresholds = {}
print(f'\nCalibrating — F1-optimal threshold per class')
print(f'  (TPR ≥ {TARGET_TPR*100:.0f}% and FPR ≤ {MAX_FPR*100:.0f}% reported as guidance only)')
print(f'{"Class":<12} {"Threshold":>10} {"FPR":>8} {"TPR":>8} {"Prec":>8} {"F1":>8}')
print('-' * 60)

for i, lbl in enumerate(LABELS):
    pos_mask = y_true_val[:, i] == 1
    neg_mask = ~pos_mask
    preds    = y_pred_val[:, i]

    best_t   = MIN_THRESHOLD
    best_f1  = 0.0
    best_stats = (1.0, 0.0, 0.0, 0.0)  # fpr, tpr, prec, f1

    for t in np.arange(MIN_THRESHOLD, 0.975, 0.005):
        tp   = np.sum(preds[pos_mask] >= t)
        fn   = np.sum(preds[pos_mask] <  t)
        fp   = np.sum(preds[neg_mask] >= t)
        tn   = np.sum(preds[neg_mask] <  t)
        tpr  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * prec * tpr / (prec + tpr) if (prec + tpr) > 0 else 0.0

        if f1 > best_f1:
            best_f1    = f1
            best_t     = float(t)
            best_stats = (fpr, tpr, prec, f1)

    thresholds[lbl] = round(best_t, 3)
    fpr, tpr, prec, f1 = best_stats
    tpr_flag = '  ⚠ low TPR' if tpr < TARGET_TPR else ''
    fpr_flag = '  ⚠ high FPR' if fpr > MAX_FPR else ''
    print(f'{lbl:<12} {best_t:>10.3f} {fpr*100:>7.2f}% {tpr*100:>7.2f}% {prec*100:>7.2f}% {f1*100:>7.2f}%{tpr_flag}{fpr_flag}')

print(f'\nCalibrated thresholds: {thresholds}')

In [ ]:
# ── Cell 7: Evaluation (test set) ─────────────────────────────────────────────
print('Running inference on test set …')
y_pred_test = model.predict(test_ds, verbose=1)
y_true_test = df_test[LABELS].values.astype('float32')

print(f'\n{"Class":<12} {"Threshold":>10} {"FPR":>8} {"TPR":>8} {"Prec":>8} {"F1":>8}')
print('-' * 64)
for i, lbl in enumerate(LABELS):
    t = thresholds[lbl]
    pos_mask = y_true_test[:, i] == 1
    neg_mask = ~pos_mask
    preds = y_pred_test[:, i]

    tp = np.sum(preds[pos_mask] >= t)
    fn = np.sum(preds[pos_mask] <  t)
    fp = np.sum(preds[neg_mask] >= t)
    tn = np.sum(preds[neg_mask] <  t)

    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    tpr  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1   = 2*prec*tpr / (prec + tpr) if (prec + tpr) > 0 else 0.0
    print(f'{lbl:<12} {t:>10.3f} {fpr*100:>7.2f}% {tpr*100:>7.2f}% {prec*100:>7.2f}% {f1*100:>7.2f}%')

# Overall multi-label accuracy
y_pred_bin  = (y_pred_test >= np.array([thresholds[l] for l in LABELS])).astype('float32')
exact_match = np.mean(np.all(y_pred_bin == y_true_test, axis=1))
hamming_acc = np.mean(y_pred_bin == y_true_test)
print(f'\nExact-match accuracy : {exact_match*100:.2f}%')
print(f'Hamming accuracy     : {hamming_acc*100:.2f}%')

In [ ]:
# ── Cell 8: Save artefacts ────────────────────────────────────────────────────
final_model_path = f'{OUT_DIR}/clothing_classifier_v5.keras'
thresholds_path  = f'{OUT_DIR}/thresholds_v5.json'

model.save(final_model_path)
print(f'Model saved → {final_model_path}')

payload = {
    'thresholds': thresholds,
    'classes': LABELS,
    'model_version': 'efficientnetb3-multilabel-v5',
    'img_size': IMG_SIZE,
    'class_weights': CLASS_WEIGHTS,
    'notes': 'Multi-label sigmoid. 40K cap. Class-weighted loss (outerwear 2.5x, footwear 1.8x). Target TPR ≥ 92%.'
}
with open(thresholds_path, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'Thresholds saved → {thresholds_path}')
print(json.dumps(payload, indent=2))

## Deployment checklist

1. Download `clothing_classifier_v5.keras` and `thresholds_v5.json` from Kaggle output.
2. Copy to `backend/model/`.
3. Update `backend/app/core/config.py`:
   ```python
   CLOTHING_MODEL_PATH     = 'model/clothing_classifier_v5.keras'
   CLOTHING_THRESHOLD_PATH = 'model/thresholds_v5.json'
   ```
4. Update `clothing_detector.py` model_version string to `efficientnetb3-multilabel-v5`.
5. `git add -f backend/model/clothing_classifier_v5.keras`
6. Commit, push to main, then `bash scripts/push-hf.sh`.